In [1]:
import pandas as pd
import os
from sqlalchemy import create_engine


DB_USER = "postgres"  
DB_PASSWORD =  os.getenv("DB_PASS")
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "BrizelOliset_db" 


engine = create_engine(
    f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("Connection of database successfully") 


Connection of database successfully


In [2]:
orders = pd.read_sql("SELECT * FROM olist_orders_dataset", con =engine)
customers = pd.read_sql("SELECT * FROM olist_customers_dataset", con=engine)
items = pd.read_sql("SELECT * FROM olist_order_items_dataset", con=engine)
payments = pd.read_sql("SELECT * FROM olist_order_payments_dataset", con=engine)

print(f"Orders shape:  {orders.shape}")
print(f"Customers shape:  {customers.shape}")
print(f"Items shape:  {items.shape}")
print(f"Payments shape:  {payments.shape}")


Orders shape:  (99441, 8)
Customers shape:  (99441, 5)
Items shape:  (112650, 7)
Payments shape:  (103886, 5)


In [3]:
#ِAggregation of one order    كل طلبه فريد من نوعه له اجمالي العناصر و اجمالي الاسعار وتكلفة الشحن الكلية 
all_items = (
    items.groupby("order_id")
    .agg(total_items=("order_item_id" , "count"),
         total_price=("price","sum"),
         total_freight=("freight_value", "sum")
         )
    .reset_index()

)

all_payments= (
    payments.groupby("order_id")
    .agg(
        total_payment=("payment_value","sum"),
        max_installments=("payment_installments",max),  
    )
    .reset_index()
)

print(f" aggreated Items shape : {all_items.shape}")
print(f" aggreagted Payments shap:  {all_payments.shape}")

 aggreated Items shape : (98666, 4)
 aggreagted Payments shap:  (99440, 3)


In [4]:
# دمج الجدول الناتج في الخلية السابقة مع جدول العميل و المدفوعات
FinalTable =orders.merge(
    customers[["customer_id","customer_state","customer_city"]],             #اعمدة محددة فقط من الجدول  والذي يهم هو id, state, city
    on="customer_id",                                                        # مفاح للربط بين الجولبن
    how="left",                                                               # left join تعني 

)

FinalTable = FinalTable.merge(all_items , on="order_id" , how="left")
FinalTable = FinalTable.merge(all_payments , on="order_id" , how="left")

FinalTable.head(8)                                   # اول 8 اسطر   صفوف 

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_state,customer_city,total_items,total_price,total_freight,total_payment,max_installments
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,SP,sao paulo,1.0,29.99,8.72,38.71,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,BA,barreiras,1.0,118.70,22.76,141.46,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,GO,vianopolis,1.0,159.90,19.22,179.12,3.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,RN,sao goncalo do amarante,1.0,45.00,27.20,72.20,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,SP,santo andre,1.0,19.90,8.72,28.62,1.0
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01,PR,congonhinhas,1.0,147.90,27.36,175.26,6.0
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaT,NaT,2017-05-09,RS,santa rosa,1.0,49.90,16.05,65.95,1.0
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07,RJ,nilopolis,1.0,59.99,15.17,75.16,3.0


In [25]:
# وجود صف واحد لكل طلب فريد
total_rows=len(FinalTable)
unique_orders =FinalTable["order_id"].nunique()
print(f"All numbers of rows : {total_rows}")
print(f"All numbers of orders : {unique_orders}")

assert(
    total_rows == unique_orders,           # اذا يوجد تكرار اظهر هذه الرسالةة
    "There is an order_id repeated"
)


All numbers of rows : 99441
All numbers of orders : 99441


<>:7: SyntaxWarning: assertion is always true, perhaps remove parentheses?
<>:7: SyntaxWarning: assertion is always true, perhaps remove parentheses?
C:\Users\HADI\AppData\Local\Temp\ipykernel_27416\104382516.py:7: SyntaxWarning: assertion is always true, perhaps remove parentheses?
  assert(


In [15]:
os.makedirs("artifacts", exist_ok=True)
artifact_path="artifacts/joined_FinalTable.csv"
FinalTable.to_csv(artifact_path, index=False)
print(f"Complete....To Save In Artifact : {artifact_path}")

Complete....To Save In Artifact : artifacts/joined_FinalTable.csv
